# Life, Death, and Biological Definition Workflow

This notebook scaffold supports the article **Life, Death, and the Problem of Biological Definition**. It can be expanded with viability decay, dormancy modeling, host-virus dynamics, borderline-case scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
viability = pd.read_csv(article_dir / 'data' / 'viability_observations.csv')
rows = []
for condition, group in viability.groupby('condition'):
    slope, intercept = np.polyfit(group['time_h'], np.log(group['live_cells']), 1)
    k = -slope
    rows.append({'condition': condition, 'loss_rate_per_h': k, 'L0': np.exp(intercept), 'half_life_h': np.log(2) / k})
pd.DataFrame(rows).round(5)

In [ ]:
cases = pd.read_csv(article_dir / 'data' / 'borderline_cases.csv')
weights = pd.read_csv(article_dir / 'data' / 'life_criteria_weights.csv')
w = dict(zip(weights['criterion'], weights['weight']))
cases['heuristic_life_score'] = sum(cases[col] * weight for col, weight in w.items())
cases.sort_values('heuristic_life_score', ascending=False).round(3)

In [ ]:
time = np.arange(0, 20.01, 0.01)
D = np.zeros_like(time)
A = np.zeros_like(time)
dead = np.zeros_like(time)
D[0] = 1e6
m = 0.02
alpha = 0.05
for i in range(1, len(time)):
    dt = time[i] - time[i-1]
    dD = -(m + alpha) * D[i-1]
    dA = alpha * D[i-1]
    dDead = m * D[i-1]
    D[i] = max(D[i-1] + dD * dt, 0)
    A[i] = A[i-1] + dA * dt
    dead[i] = dead[i-1] + dDead * dt
pd.DataFrame({'time': time, 'dormant': D, 'active': A, 'dead_or_lost': dead}).tail().round(3)